In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os

file_path = os.path.expanduser("~/Desktop/regge.xlsx")
df = pd.read_excel(file_path)

model_trust = df[
    ["Preis", "Seller Rating", "Seller Type", "zahl der verkauften artikel"]
].copy()

def clean_price(x):
    if pd.isna(x):
        return np.nan
    x = str(x).replace("EUR", "").replace("€", "").replace(".", "").replace(",", ".").strip()
    try:
        return float(x)
    except:
        return np.nan

def clean_rating(x):
    if pd.isna(x) or x == "N/A":
        return np.nan
    x = str(x).replace("%", "").replace(",", ".").strip()
    try:
        return float(x)
    except:
        return np.nan

def clean_number(x):
    if pd.isna(x) or x == "N/A":
        return np.nan
    x = str(x).replace(".", "").replace(",", ".").replace("+", "").strip()
    try:
        return float(x)
    except:
        return np.nan

model_trust["price"] = model_trust["Preis"].apply(clean_price)
model_trust["seller_rating"] = model_trust["Seller Rating"].apply(clean_rating)
model_trust["seller_sales_count"] = model_trust["zahl der verkauften artikel"].apply(clean_number)

model_trust = model_trust.drop(columns=["Preis", "Seller Rating", "zahl der verkauften artikel"])
model_trust = model_trust.dropna()

model_encoded = pd.get_dummies(
    model_trust,
    columns=["Seller Type"],
    drop_first=True
)

X = model_encoded.drop("price", axis=1)
y = model_encoded["price"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)

trust_model = sm.OLS(y, X).fit()

trust_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  price   R-squared:                       0.172
Model:                            OLS   Adj. R-squared:                  0.159
Method:                 Least Squares   F-statistic:                     13.47
Date:                Sat, 02 May 2026   Prob (F-statistic):           5.05e-08
Time:                        17:15:03   Log-Likelihood:                -1153.1
No. Observations:                 199   AIC:                             2314.
Df Residuals:                     195   BIC:                             2327.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
======================================================================================
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                801.8865     11.732     68.351      0.000     778.749     825.024
seller_rating          0.4275      0.138      3.089      0.002       0.155       0.700
seller_sales_count  2.928e-05   1.02e-05      2.863      0.005    9.11e-06    4.94e-05
Seller Type_Privat   -21.5565     14.037     -1.536      0.126     -49.241       6.128
==============================================================================
Omnibus:                       27.574   Durbin-Watson:                   1.506
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               37.895
Skew:                           0.842   Prob(JB):                     5.90e-09
Kurtosis:                       4.317   Cond. No.                     1.81e+06
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.81e+06. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [2]:
print("Observations:", len(y))
print("R²:", trust_model.rsquared)
print("Adjusted R²:", trust_model.rsquared_adj)

Observations: 199
R²: 0.1716173376829705
Adjusted R²: 0.15887298903193936
